# 04 · Price optimization (NLopt)

Phase 4 results: optimal **regular** prices for the portfolio, solved for **two goals**:

- **profit:** maximise `Σ (P − c)·Q`;
- **revenue:** maximise `Σ P·Q`, which is the same formula with cost = 0.

For each product *i*, with current regular price P₀ and weekly baseline volume B (both from the last 13 weeks), regular-price elasticity β (Phase 2) and unit cost c = (1 − category margin)·P₀:

- **demand:** `Q_i(P) = B_i · (P / P₀_i)^β_i`, the constant-elasticity curve implied by the log-log model;
- **constraints**, all written as `fc ≤ 0`:
  - prices within 0.8–1.2 × P₀;
  - portfolio volume ≥ 95% of current;
  - revenue-weighted average price change ≤ +5%;
  - private-label ladders kept.

The problem is solved with NLopt (SLSQP, compared with COBYLA and ISRES). Prices are rounded to "9" endings, repaired so every constraint still holds, and checked by a **safeguard** that keeps current prices if rounding would score below them. Each product also gets a **rule check**: its own optimum is `P* = c_eff·β/(1+β)`, so for revenue the classic rule applies (elastic → cut, inelastic → raise), and for profit the rule is margin-aware (raise when |β| < 1/margin).

Logic lives in `src/optimization/`; run `make optimize` first.

In [ ]:
# reload edited src/ modules automatically (avoids stale imports after code changes)
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

from src.config import FIGURES, OPTIMIZATION_SUMMARY_PATH, RECOMMENDATIONS_PATH, Config
from src.plots import FigureStore, OptimizationPlotter, PlotStyle
from src.schema import Col, Metric, Scenario, kpi_col

style = PlotStyle()
style.apply()
plotter, store = OptimizationPlotter(style), FigureStore(FIGURES)
bounds = Config.load().optimization.price_bounds
all_recs = pd.read_parquet(RECOMMENDATIONS_PATH)
all_summary = pd.read_parquet(OPTIMIZATION_SUMMARY_PATH)

## Choose the goal
Set `objective` to `"profit"` or `"revenue"` and re-run the cells below. Every table and chart then shows that goal's results. KPIs always report real profit (at unit cost), whatever the goal.

In [ ]:
objective = "profit"  # "profit" or "revenue"
recs = all_recs[all_recs[Col.OBJECTIVE] == objective]
summary = all_summary[all_summary[Col.OBJECTIVE] == objective]

## Profit goal vs revenue goal
Recommended (rounded) prices for each goal, compared with current prices. Maximising revenue cuts prices on elastic products: it buys volume and revenue but gives up profit.

In [ ]:
rec = all_summary[all_summary[Col.SCENARIO] == Scenario.RECOMMENDED.value]
rec.set_index(Col.OBJECTIVE)[
    ["volume_change", "revenue_change", "profit_change", "annual_profit_delta"]
].round(3)

In [ ]:
all_recs.groupby([Col.OBJECTIVE, Col.RULE_STATUS]).size().unstack(fill_value=0)

## Solvers and scenarios (selected goal)
`recommended` is the SLSQP optimum rounded to price points; `linear_formula` is the linear-demand optimum scored on the constant-elasticity curve.

In [ ]:
cols = [Col.SCENARIO, "volume_change", "revenue_change", "profit_change", "annual_profit_delta"]
summary[[*cols, Col.EVALUATIONS, Col.FIT_SECONDS, Col.STATUS]].round(4)

In [ ]:
store.save(plotter.scenarios(summary), f"04_scenarios_{objective}")

## Recommended price changes
The dot is the recommended change; the line is the 5th–95th percentile of the optimal price across elasticity draws. A wide line means the elasticity is uncertain, so the price move should be tested before rollout.

In [ ]:
store.save(plotter.price_changes(recs, bounds), f"04_price_changes_{objective}")

## One product, explained
The curve is the selected goal's weekly value against price. The product's own optimum (▲) can differ from the portfolio recommendation (◆): the +5% average-increase cap makes the optimizer trade price moves across products. The dashed curve is the linear formula: close to the current price, but wrong away from it.

In [ ]:
store.save(plotter.objective_curve(recs, 2066200531, bounds), f"04_objective_curve_{objective}")

### Explore any product
1. Run the next cell to list every product with its UPC, elasticity, recommended change and rule check. Filter it, for example `products[products[Col.DESCRIPTION].str.contains("CHEERIOS")]`.
2. Copy a UPC into `upc = ...` in the cell after, and run it. The chart is shown in the notebook only, not saved.

In [ ]:
cols = [Col.UPC, Col.DESCRIPTION, Col.PRODUCT_SIZE, Col.CATEGORY, Col.ELASTICITY, Col.PRICE_RATIO]
products = recs[[*cols, Col.EXPECTED_DIRECTION, Col.RULE_STATUS]]
products.sort_values([Col.CATEGORY, Col.DESCRIPTION]).round(3)

In [ ]:
upc = 1111009477  # change to any UPC from the list above
plotter.objective_curve(recs, upc, bounds)

## Recommendation table and rule check
`expected_direction` is where the product's own optimum lies under the selected goal. `constraint_override` means a portfolio constraint (cap, ladder or volume floor) moved it the other way.

In [ ]:
price, profit = (
    kpi_col(Scenario.RECOMMENDED, Metric.PRICE),
    kpi_col(Scenario.RECOMMENDED, Metric.PROFIT),
)
view = recs.assign(profit_change=recs[profit] / recs[kpi_col(Scenario.CURRENT, Metric.PROFIT)] - 1)
cols = [
    Col.DESCRIPTION,
    Col.PRODUCT_SIZE,
    Col.ELASTICITY,
    Col.CURRENT_PRICE,
    price,
    Col.PRICE_RATIO,
]
checks = [Col.EXPECTED_DIRECTION, Col.ACTUAL_DIRECTION, Col.RULE_STATUS, "profit_change"]
view[[*cols, *checks]].sort_values(Col.PRICE_RATIO).round(3)